**注意力机制（Attention Mechanism）**

从最初的机器翻译到如今的大模型、自动驾驶和多模态，注意力机制衍生出了庞大的变种家族。按其**底层数学原理、计算区域和稀疏策略**，我们可以将所有主流的注意力变种划分为以下几大门派：

---

## 1. 经典与基础注意力（Foundational Attention）

在 Transformer 诞生之前或早期，奠定整个机制基石的经典操作。

* **加性注意力 (Additive / Bahdanau Attention)**: 早期用于 RNN 机器翻译。通过一个小型全连接网络（$\tanh$ 激活）来计算 Query 和 Key 的相关性，计算量较大。
* **点积注意力 (Dot-Product / Luong Attention)**: 直接计算 $Q K^T$。计算速度快，矩阵乘法对硬件极友好。
* **缩放点积注意力 (Scaled Dot-Product Attention)**: Transformer 的奠基核心。在点积的基础上除以 $\sqrt{d_k}$（特征维度的根号），防止维度过高时点积结果过大、导致 Softmax 梯度消失。
* **自注意力 (Self-Attention)**: $Q, K, V$ 均源自同一个输入序列，用于捕获序列内部的自身依赖关系。
* **交叉注意力 (Cross / Encoder-Decoder Attention)**: $Q$ 来自一个序列（如 Decoder），而 $K, V$ 来自另一个序列（如 Encoder），用于跨模态或跨序列的信息对齐。

---

## 2. 软硬件工程落地与计算降维（Efficient & Linear Attention）

为了打破 $O(N^2)$ 的时空复杂度，用数学近似或硬件优化改写注意力计算公式。

* **线性注意力 (Linear Attention)**: 利用核函数（Kernel Function）的结合律，将计算顺序由 $(Q K^T) V$ 变为 $Q (K^T V)$，使时间复杂度直接退化为线性 $O(N)$。
* **随机特征注意力 (Random Feature Attention / FAVOR+)**: Performer 的核心。利用正交随机特征来无损近似标准 Softmax 的指数映射，既维持了标准注意力的表达力，又实现了线性复杂度。
* **低秩投影注意力 (Low-rank Projection Attention)**: Linformer 的核心。在计算 $Q K^T$ 前，通过固定矩阵将 $K$ 和 $V$ 的序列长度维度投影到极小的常数 $k$，实现线性化。
* **FlashAttention (闪存注意力)**: 工业界的绝对统治者。**不改变任何数学公式**，而是从硬件底层出发，利用 GPU 的 SRAM 缓存分块（Tiling）计算，动态在线（Online）更新 Softmax，消除了读写 HBM 显存的瓶颈。

---

## 3. 工业大模型工程化变种（Multi-Head Variants）

针对大模型大并发推理（Generation）时，因缓存 $K, V$ 矩阵（KV Cache）导致显存爆炸而做的结构裁剪。

* **多头注意力 (MHA, Multi-Head Attention)**: 标准形态。每个 Head 拥有自己独立的 $Q, K, V$ 权重，让模型能从多个子空间学习不同尺度的特征。
* **多查询注意力 (MQA, Multi-Query Attention)**: 极端剪枝。所有 Head **共享同一组 $K$ 和 $V$ 权重**，只有 $Q$ 保持多头。推理时 KV Cache 显存占用骤降，但模型表达力会有轻微跌落。
* **分组查询注意力 (GQA, Grouped-Query Attention)**: 折中与完美方案（LLaMA 3 标配）。将 Query 头分组，**每组 Query 共享一组 $K, V$ 头**。在 MHA 的高容量和 MQA 的低显存之间取得了工业最佳平衡。

---

## 4. 空间稀疏与模式裁剪（Sparse & Windowed Attention）

通过人工施加“先验限制”，规定 Token 只能看到特定范围的邻居，从而强制实现稀疏计算。

* **局部滑动窗口注意力 (Sliding Window / Local Attention)**: 每个 Token 只能关注其左右固定窗口大小（如 $w$）内的邻居，复杂度降为 $O(N \cdot w)$。
* **膨胀/空洞注意力 (Dilated Window Attention)**: 类似于空洞卷积（Atrous Convolution）。在滑动窗口的基础上跳跃式采样（如每隔 2 个 Token 采样一次），在不增加计算量的前提下成倍放大感受野。
* **全局轴向注意力 (Global-Local / Axial Attention)**: 允许特定关键 Token（如 `[CLS]` 或句首 Token）拥有全局注意力，其余 Token 仅保留局部滑动窗口。
* **块状注意力 (Blockwise / Chunked Attention)**: 将超长序列强制切分成不重叠的固定尺寸 Block，只在 Block 内部计算完整的 Self-Attention，切断跨 Block 的计算。

---

## 5. 动态采样与几何拓扑（Dynamic & Graph Attention）

打破刚性网格，让注意力根据数据内容动态决定“去哪里看、看多远”。

* **可变形注意力 (Deformable Attention)**: 视觉领域的重大突破。模型基于输入的特征生成一组**偏移量（Offsets）**，在一个全局参考点周围动态采样极少数的关键点（通常只有 4 个或 8 个）来计算注意力，是自动驾驶 BEV（鸟瞰图）感知网络的骨架。
* **图注意力 (GAT, Graph Attention)**: 专门处理非欧几何图结构。节点的注意力权重不再基于序列位置，而是基于图的邻接矩阵拓扑关系，仅对存在边（Edge）连接的邻居节点聚合信息。
* **交叉尺度注意力 (Cross-Scale Attention)**: 在视觉大模型中，用于同时捕获高分辨率（细粒度局部）和低分辨率（粗粒度全局）特征图之间的空间相关性。
* **时空分离注意力 (Divided Space-Time Attention)**: 视频生成（如 Sora、Vivit）的核心。将时空全连接注意力拆解，先做帧内空间注意力（Spatial Attention），再做跨帧沿时间轴的时间注意力（Temporal Attention）。

---

## 6. 特殊算子与另类泛化（Alternative Attentions）

虽然叫“注意力”，但已经脱离了标准点积和 Softmax 的范畴，通过引入其他数学工具实现信息交互。

* **向量注意力 (Vector Attention)**: 点积注意力的输出是一个标量权重；而向量注意力产生的是一个**与特征通道同维度的权重向量**，能在通道级别施加动态调控，在 3D 点云网络（Point Transformer）中非常强大。
* **傅里叶/频域注意力 (Fourier/Frequency Attention)**: 将输入映射到频域，通过频域的点乘（等价于时域的卷积）来实现全局 Token 的混合，计算复杂度为极低的 $O(N \log N)$。
* **拉普拉斯/高斯注意力 (Laplacian / Gaussian Attention)**: 在计算注意力矩阵时，隐式地根据物理距离注入高斯或拉普拉斯衰减偏置，强制让模型具备“越近越重要”的强归纳偏置。

---

### 🛠️ 终极总结：注意力演进的底层逻辑

如果我们把所有的变种抽丝剥茧，所有研究者在魔改注意力时，本质上都在做这三件事的权衡（Trade-off）：

1. **全局感知的妥协**：标准注意力是“人人看我，我看人人”（$O(N^2)$）。变种注意力要么通过**窗口**限制范围，要么通过**投影**降低分辨率，要么通过**动态采样**挑着看。
2. **工程与硬件的妥协**：数学上再精妙的变种（如各种哈希、核函数），如果在 GPU 上需要频繁读写低速显存，速度反而不如纯粹靠硬件分块、暴力计算的 **FlashAttention**。
3. **计算参数的复用**：从 MHA $\rightarrow$ GQA $\rightarrow$ MQA，本质上是在用“Key-Value 特征的通道复用”，来换取推理阶段极度紧缺的显卡带宽。

Transformer 自 2017 年问世以来，为了解决其**计算复杂度高（$O(N^2)$）**、**长文本处理能力弱**、缺乏归纳偏置（Inductive Bias）等问题，衍生出了极其庞大的变种家族（通常被称为 **X-formers**）。

为了让你对整个大家族有个清晰的脉络，我将这些主流变种按**核心改进方向**进行了归类梳理：

---

## 1. 计算效率与线性化变种（Linear & Efficient Transformers）

这类变种的核心目标是打破传统 Attention 的 $O(N^2)$ 时间和空间复杂度，使其能够处理超长序列。

* **Linformer**: 通过低秩矩阵分解（Low-rank Projection），将 Key 和 Value 投影到超短维度，将复杂度降为 $O(N)$。
* **Performer**: 利用正交随机特征（Fast Attention Via Positive Random Features, FAVOR+）来近似 Softmax Attention，实现真正的线性复杂度。
* **Linear Transformer**: 移除 Softmax，利用核函数（Kernel Function）的结合律先计算 $K^T V$，从而实现 $O(N)$ 复杂度。
* **Reformer**: 引入局部敏感哈希（LSH, Locally Sensitive Hashing）将相似的 Token 聚类，只在同组内计算 Attention；同时使用可逆残差网络（Reversible Residual Layers）节省显存。
* **FlashAttention (1, 2, 3)**: *严格来说是硬件加速层面的变种*。它不改变 Attention 的数学本质，而是通过融合 GPU 算子、利用 SRAM 高速缓存和重计算（Recomputation），极大提升了传统 Attention 的计算速度并降低了显存占用。

---

## 2. 稀疏注意力变种（Sparse Attention Transformers）

通过限制每个 Token 只能看到部分指定区域的 Token，将密集计算稀疏化。

* **Sparse Transformer**: 引入阶乘步长（Strided）和局部固定模式的稀疏注意力，复杂度降为 $O(N\sqrt{N})$。
* **Longformer**: 结合了**局部滑动窗口注意力（Sliding Window）**和**全局注意力（Global Attention）**（如针对 `[CLS]` 标记），适合处理长文本。
* **BigBird**: 在 Longformer 的基础上，增加了**随机注意力（Random Attention）**，在理论上证明了稀疏注意力仍能保持全连接 Attention 的表达能力。

---

## 3. 长文本与序列长度拓展变种（Long-context & Recurrent Transformers）

改变位置编码或引入隐状态，使模型能处理甚至“无限”延伸的流式序列。

* **Transformer-XL**: 引入**片段循环机制（Segment-level Recurrence）**和**相对位置编码（Relative Position Encoding）**，允许模型缓存上一个片段的隐状态，突破了固定长度窗口的限制。
* **Compressive Transformer**: 在 Transformer-XL 的基础上，对更久远的过去记忆进行压缩（类似于细粒度记忆和粗粒度记忆），进一步拉长注意力窗口。
* **RoFormer**: 引入了**旋转位置编码（RoPE, Rotary Position Embedding）**。虽然它本身是一个位置编码变种，但它是当今大模型（LLaMA, Mistral 等）外推序列长度（Context Window Extension）的基石。

---

## 4. 视觉与多模态变种（Vision & Multimodal Transformers）

将 Transformer 的架构从 NLP 领域成功跨界到 CV 及多模态领域。

* **ViT (Vision Transformer)**: 奠基之作。将图像切分成不重叠的 Patch（如 16x16），把每个 Patch 视作一个词（Token）直接送入标准 Transformer Encoder。
* **Swin Transformer**: 引入**滑动窗口（Shifted Windows）**和**层次化表征（Hierarchical Feature Maps）**。将计算限制在局部窗口内以实现线性复杂度，同时通过窗口滑动实现跨窗口交互，成为 CV 领域的骨干网络（Backbone）。
* **DeiT (Data-efficient Image Transformers)**: 引入了教师-学生蒸馏策略（Distillation Token），解决了 ViT 极度依赖海量预训练数据的问题。
* **Crossformer / CrossViT**: 采用双分支或跨尺度注意力，专门用来融合不同大小（Scale）的图像 Patch 信息。

---

## 5. 结构与轻量化变种（Structural & Lightweight Transformers）

改变标准的层级堆叠方式，或与其他架构（如 CNN、RNN）融合，追求更低的参数量或更强的感知识别能力。

* **Universal Transformer**: 结合了 RNN 的循环特性。在深度方向上**共享同一层 Transformer 的参数**，动态决定计算步数（Adaptive Computation Time），参数量极小。
* **Conformer**: 将 **Convolution（卷积）** 和 **Transformer** 结合。利用 Transformer 捕获全局长距离依赖，利用 CNN 捕获局部特征，在语音识别（ASR）领域表现极佳。
* **MobileViT**: 为移动端设计的轻量化视觉变种，将 MobileNet 卷积的高效性与 ViT 的全局全局建模能力相融合。

---

## 6. 当前大模型（LLM）主流演进变种

在实际的大语言模型落地中，由于硬件和吞吐量的考量，目前最普及的变种主要集中在 **Attention 机制的简化**上：

* **MHA (Multi-Head Attention)**: 标准的多头注意力，每个 Head 都有独立的 $K$ 和 $V$。
* **MQA (Multi-Query Attention)**: 所有的 Head **共享同一组 $K$ 和 $V$**，极大减少了推理（Generation）阶段 KV Cache 的显存占用。
* **GHA / GQA (Grouped-Query Attention)**: MHA 和 MQA 的折中方案。将 Query 分组，**每组 Query 共享一组 $K$ 和 $V$**（例如 LLaMA 3 采用的架构），在速度和模型容量之间取得了完美的平衡。

---

## 7. 动态与可变形注意力变种（Dynamic & Deformable Transformers）

这类变种的核心思想是**打破固定的网格或全局采样**，让注意力机制自己去“寻找”最关键的区域，动态调整计算位置。

* **Deformable Transformer (可变形 Transformer)**: 针对 ViT 和 DETR 在高分辨率图像上计算量爆炸的问题而生。它不再让一个 Token 和全图所有 Token 计算注意力，而是仿照可变形卷积（DCN），**只在参考点周围动态采样少量的关键点（Key points）**。复杂度直接从全局的 $O(N^2)$ 降为局部线性的 $O(N)$，是现代自动驾驶感知算法（如 BEVFormer 等）的绝对核心底层。
* **Dynamic Vit / PSViT**: 根据图像内容动态识别并剪枝（Pruning）掉不重要的背景 Token，让计算资源集中在目标物体对应的 Token 上，实现动态加速。

---

## 8. 空间-时间与三维感知变种（Spatiotemporal & 3D Transformers）

专门用于处理视频、点云（PointCloud）、雷达（LiDAR）等多维时空数据的变种，在自动驾驶和视频生成（如 Sora 架构底层）中应用极广。

* **Timesformer / Vivit**: 视频领域的里程碑。为了处理视频的“空间+时间”维度，它们采用了**时空分离注意力（Space-Time Separated Attention）**：先在同一帧内计算空间注意力，再在不同帧的同一位置计算时间注意力，避免了直接计算时空全连接导致的算力崩溃。
* **Point Transformer (v1, v2, v3)**: 专为 3D 点云数据设计。利用向量注意力（Vector Attention）和 K 近邻（KNN）局部图结构，让 Transformer 能够完美处理无序、稀疏的三维点云数据。
* **Voxel Transformer (VoxFormer)**: 将三维空间体素化（Voxelization），并在体素空间内设计稀疏的注意力机制，常用于自动驾驶中的三维场景重建和语义分割。

---

## 9. 现代大模型（LLM）底层架构与工程进化变种

伴随着 ChatGPT、LLaMA 等大模型的爆发，近两年来在底层网络拓扑、归一化和门控机制上出现了一批标准配置变种。

* **SwiGLU Transformer**: 现代大模型的标准 FFN（前馈网络）变种。将传统的 ReLU 或 GELU 激活函数替换为 **SwiGLU（Swish Gated Linear Unit）**。虽然增加了参数量和计算量，但表征能力大幅提升，LLaMA、PaLM 等无一例外全部采用。
* **Root Mean Square Normalization (RMSNorm)**: 标准 Transformer 使用 LayerNorm（包含均值和方差归一化）。RMSNorm 认为只做均方根归一化（不减去均值）就能达到同样的效果，同时能带来 10%~50% 的层间计算加速。
* **DeepNorm / SandWich Norm**: 针对超深 Transformer（如大于 100 层）在训练初期极易梯度爆炸或消失的问题，通过微调残差连接的缩放比例和归一化位置，保证上百层的模型能稳定收敛。

---

## 10. 突破 Self-Attention 的“平替”与泛化变种（Beyond Attention）

严格来说，这属于 **Transformer 的泛化或平替架构**。它们保留了 Transformer 的整体 Block 结构（残差、FFN、LayerNorm），但**把最核心的 Self-Attention 替换成了其他数学算子**，以追求更高的效率。

* **Fourier Former (FNet)**: 谷歌提出的一种极端激进的变种。它直接**把 Self-Attention 换成了二维离散傅里叶变换（DFT）**！完全没有训练参数，却能实现 Token 之间的全连接信息混合，速度极快，且能保留标准 Transformer 90% 以上的性能。
* **PoolFormer**: 学术界的一个经典反思。它证明了 Transformer 之所以强大，很大程度上得益于其“MetaFormer”的整体元架构（Token 混合 + Channel 混合 + 残差）。PoolFormer 直接**用最简单的池化（Pooling）算子代替 Attention**，在 CV 任务上依然刷榜。
* **Mamba / State Space Models (SSM)**: 严格来说它是 Transformer 的强力竞争者。它引入了时变状态空间模型，虽然不是 Attention，但它通过仿 Transformer 的层级堆叠，实现了**上下文吞吐量无上限、推理时间复杂度 $O(1)$** 的惊人性能，目前正与 Transformer 激烈交锋。

---

## 11. 图形与树状网络变种（Graph & Structure-Aware Transformers）

将 Transformer 的全局关联能力，引入到非欧几里得空间（如社交网络、分子结构、代码树）中。

* **Graph Transformer / Graphormer**: 传统的 GNN（图神经网络）容易产生过度平滑（Over-smoothing），Graphormer 将图的**中心度编码（Centrality Encoding）**、空间编码（Spatial Encoding）**和**边编码（Edge Encoding）以偏置项的形式注入到 Attention 矩阵中，让标准 Transformer 能够完美理解图的拓扑结构。
* **Tree Transformer**: 专为自然语言的语法树或代码的抽象语法树（AST）设计，通过限制注意力汇聚的方向，使其严格按照树状层级结构进行前向传播。

---

## 12. 跨模态多分支与联合注意力变种（Multi-Branch & Joint Attention Transformers）

传统的 Transformer 要么只处理文本（GPT），要么只处理图像（ViT），要么用简单的 Cross-Attention 连起来。而新一代的多模态大模型，直接在 Transformer Block 的**内部通路**上做文章。

* **MMDiT (Multimodal Diffusion Transformer)**: 也就是你提到的、由 Stable Diffusion 3 首次采用的核心架构。
* *核心痛点*：在传统 DiT 中，文本和图像的概念差异极大，强行压到一个线性空间或者用单向 Cross-Attention 会导致文本控制力不足、文字排版（Spelling）稀烂。
* *底层魔改*：它在同一个 Block 内部设计了**两个独立的 Transformer 分支（拥有独立的权重）**，分别处理图像 Token 和文本 Token；但在**计算 Attention 的时候，它把两者的序列拼接在一起，进行联合注意力（Joint Attention）计算**。这样既让图像和文本在各自的通道空间进化，又实现了完全平等的双向信息对齐。


* **Dual-Stream / Dual-Branch Transformer**: 与 MMDiT 类似，常用于早期多模态对齐模型（如 ViLBERT）。图像和文本各走一条流，在中间层插入密集的交叉注意力进行握手，最后再分开。
* **Unified / Single-Stream Transformer (如 FLUX, PixArt-α)**: 与 MMDiT 理念相对。它不分流，直接把文本 Token、音频 Token 或图像 Token 通过不同的 Linear 层投影到相同维度，然后拼成一个超长的序列，直接丢进一个纯粹的、共享权重的标准 Transformer 里面暴力计算。

---

## 13. 扩散生成专用的调制变种（Diffusion-Conditioned Transformers）

为了在去噪过程中加入“时间步（Timestep $t$）”和“各类条件（Condition）”，研究人员对 Transformer 内部的**归一化层和残差层**进行了魔改。

* **DiT (Diffusion Transformer)**: 舍弃了传统的 U-Net，把 Latent Image 变成 Patch 序列。它最核心的变种特征是引入了 **adaLN-Zero (Adaptive Layer Normalization)**：通过一个 MLP 把时间步 $t$ 和类别标签映射为缩放因子 $\gamma$ 和偏置 $\beta$，动态调制每一层 Transformer 的激活值。
* **U-ViT / UViT**: 一种将 U-Net 的 CNN 骨架彻底替换为 ViT、但保留了 U-Net 的跳跃连接（Skip Connections）的变种。它在浅层 Block 和深层 Block 之间拉了一条“小路”直接把特征拼起来，有效缓解了深层生成模型的信息丢失。

---

## 14. 细粒度、局部特征增强变种（Locality & Token-Reduction Transformers）

为了在超高分辨率图像或视频合成中节省算力，又不想失去空间连续性，研究人员对 Token 的提取和处理阶段进行了进化。

* **E-MMDiT (Efficient MMDiT)**: 这是近期针对 MMDiT 演进的轻量化变种。为了解决长 Token 导致的延迟问题，它引入了**交替子区域注意力（ASA, Alternating Subregion Attention）**和**位置强化（Position Reinforcement）**，在大幅度压缩 Token 数量的前提下，依然能维持极高画质的图像合成。
* **Segment/Patch-Merging Transformer**: 如 Swin 或 SegFormer 中的设计。在网络不断变深的过程中，它会通过一个 Patch Merging 层把相邻的 $2 \times 2$ 个 Token 融合成一个，使 Token 数量随深度递减，形成漏斗状的拓扑结构（类似于 CNN 的池化降采样）。

---

## 🔄 再次更新：从“全模态联合”重新看 Transformer 谱系

如果我们把 MMDiT、DiT、Mamba 放到一起，现在的 Transformer 种类划分实际上已经打破了 NLP 和 CV 的界限，演变成了**三大演进意识形态**：

```
                              Transformer 演进新格局
                                        │
         ┌──────────────────────────────┼──────────────────────────────┐
         ▼                              ▼                              ▼
【单模态/自回归/大语言】        【跨模态/扩散生成/多流交互】    【无 Attention 算子平替】
(LLM 阵营: 堆叠全连接)         (AIGC 阵营: 空间解耦/联合)      (新架构阵营: 线性吞吐)
  - LLaMA (MHA/GQA)               - DiT (adaLN 调制)              - Mamba (状态空间)
  - Mistral (滑动窗口)            - MMDiT (图像/文本双分支联合)    - FNet (傅里叶变换)
  - DeepSeek (MLA低秩注意力)      - FLUX (单流统一混合序列)       - RWKV (线性注意力RNN化)

```


### 🗂️ 终极全景分类矩阵

为了让你对这几十种变种有一个高屋建瓴的认识，我们可以从“如何改变 Token 交互方式”的底层逻辑来画一张全景图：

| 改进维度 | 核心手段 | 代表变种 |
| --- | --- | --- |
| **位置与采样魔改** | 动态、局部非固定采样 | **Deformable Transformer**, Swin Transformer |
| **复杂度降维 ($O(N)$)** | 低秩、核函数、哈希、硬件映射 | Linformer, Performer, Linear Transformer, **FlashAttention** |
| **时空维度拓展** | 维度分离、稀疏体素、局部邻域 | **Timesformer**, Point Transformer, VoxFormer |
| **长文本外推** | 隐状态缓存、旋转映射 | Transformer-XL, **RoFormer (RoPE)** |
| **大模型工程落地** | 算子门控、极简归一化、KV共享 | **SwiGLU**, RMSNorm, **GQA**, MQA |
| **核心算子替换** | 傅里叶变换、池化、状态空间 | FNet, PoolFormer, **Mamba (SSM)** |

从当年一统 NLP 的 `Vanilla Transformer`，到如今上天入地的各种变形，Transformer 的演进本质上是一场“在全局表征能力（Inductive Bias 极少）与计算效率（硬件实现友好度）之间寻找最佳平衡点”的漫长拉锯战。